# Combined Confidence Score for Metabolomics Identifications

**Goal:** Merge multiple independent signals into a single human-interpretable confidence score (0–1).
Each component captures one dimension of confidence; score collapses if any component fails.

**Components:**
1. **MS2 match** — entropy similarity (identity_score)
2. **Uniqueness** — gap between top hit and runner-up (sim_gap)
3. **RT agreement** — predicted vs observed retention time (delta_rt_pred)
4. **Spectrum quality** — query spectrum informativeness (entropy)

**Score formula:** multiplicative (geometric mean) → if any component ≈ 0, confidence collapses.

**Validation:** correlate with Oliver's ad hoc probability and 124 expert comments.

**Output:** absurd identifications flagged, confidence distribution plotted.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

warnings.filterwarnings('ignore')

ROOT = '/Users/ellayoung/Desktop/metabolo_confi_score'
OUT = f'{ROOT}/results/current'

# Load Orbitrap spreadsheet with Oliver's expert judgments
print('Loading data...')
raw = pd.read_excel(
    f'{ROOT}/data/masswiki_Orbitrap HILIC negESI_2026-03-19.xlsx',
    sheet_name='masswiki_result_2026-03-19',
    header=4
)

df = raw.rename(columns={
    'identity_score': 'identity',
    'fuzzy_score': 'fuzzy',
    'identity_score_reference_library_': 'ref_sim_1st',
    'reference_library_search-identity_score_2nd': 'ref_sim_2nd',
    'DRTpred': 'delta_rt_pred',
    'Unnamed: 36': 'oliver_comment',
    'oliver\'s ad hoc probability': 'oliver_prob',
})

for col in ['identity', 'fuzzy', 'ref_sim_1st', 'ref_sim_2nd', 'delta_rt_pred', 'entropy', 'oliver_prob']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove ISTDs
is_istd = df['name'].astype(str).str.startswith('1_')
df = df[~is_istd].copy()
print(f'Loaded {len(df)} spectra (ISTDs removed)')

# Load full hits for sim_gap
hits = pd.read_csv(f'{ROOT}/data/library_hits/orbitrap_hilic_neg_masswiki_hits.csv', low_memory=False)
hits['entropy_similarity'] = pd.to_numeric(hits['entropy_similarity'], errors='coerce')
print(f'Loaded {len(hits)} library hits')

## 1. Compute sim_gap from full hits

In [ ]:
from rdkit import Chem
from rdkit.Chem.inchi import InchiToInchiKey, MolToInchi

def smiles_to_inchikey14(smi):
    if not isinstance(smi, str) or not smi.strip():
        return None
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        ik = InchiToInchiKey(MolToInchi(mol))
        return ik[:14] if ik else None
    except Exception:
        return None

# Get top 2 distinct compounds per spectrum by InChIKey14
ref_hits = hits[hits['hit_source'] == 'reference'].copy()
unique_smiles = ref_hits['smiles'].dropna().unique()
smi_to_ik14 = {smi: smiles_to_inchikey14(smi) for smi in unique_smiles}
ref_hits['ik14'] = ref_hits['smiles'].map(smi_to_ik14)
ref_hits['dedup_key'] = ref_hits['ik14'].fillna(ref_hits['lib_name'])

best_per_compound = (ref_hits
    .sort_values('entropy_similarity', ascending=False)
    .groupby(['wiki_id', 'dedup_key'])
    .first()
    .reset_index()
)

ranked = (best_per_compound
    .sort_values(['wiki_id', 'entropy_similarity'], ascending=[True, False])
    .groupby('wiki_id')
)
top1 = ranked.nth(0).set_index('wiki_id')[['entropy_similarity']].rename(
    columns={'entropy_similarity': 'sim_1st'})
top2 = ranked.nth(1).set_index('wiki_id')[['entropy_similarity']].rename(
    columns={'entropy_similarity': 'sim_2nd'})

sim_gap_df = top1.join(top2, how='left')
sim_gap_df['sim_gap'] = (sim_gap_df['sim_1st'] - sim_gap_df['sim_2nd']).fillna(sim_gap_df['sim_1st'])

df = df.merge(sim_gap_df[['sim_gap']], left_on='wiki_id', right_index=True, how='left')
print(f'Computed sim_gap: {df["sim_gap"].notna().sum()} spectra')

## 2. Define component score functions (each outputs 0–1)

In [ ]:
def score_ms2_match(identity):
    """MS2 match quality: raw entropy similarity (0-1)."""
    return np.clip(identity, 0, 1)

def score_uniqueness(sim_gap):
    """Identification uniqueness: how clear is the winner from runner-up?
    Gap < 0.05 → low confidence. Gap > 0.2 → high confidence.
    Sigmoid-like: 1 - exp(-gap / scale).
    """
    if pd.isna(sim_gap):
        return 0.5  # neutral if unknown
    scale = 0.05
    return 1.0 - np.exp(-sim_gap / scale)

def score_rt_agreement(delta_rt_pred):
    """RT agreement: Gaussian penalty for deviation.
    ΔRT = 0 → 1.0 (perfect). ΔRT = 40s → ~0.14 (2 sigma away).
    """
    if pd.isna(delta_rt_pred):
        return 0.5  # neutral if unknown
    sigma_rt = 20  # assume 20s is 1 sigma of model error
    return np.exp(-((delta_rt_pred / sigma_rt) ** 2))

def score_spectrum_quality(entropy):
    """Spectrum informativeness: entropy as a measure of peak complexity.
    Entropy < 0.5 → uninformative. Entropy > 2 → highly informative.
    Linear from 0 at entropy=0 to 1 at entropy=2.
    """
    if pd.isna(entropy):
        return 0.5  # neutral if unknown
    return np.clip(entropy / 2.0, 0, 1)

# Compute all components
print('Computing confidence components...')
df['score_ms2'] = df['identity'].apply(score_ms2_match)
df['score_unique'] = df['sim_gap'].apply(score_uniqueness)
df['score_rt'] = df['delta_rt_pred'].apply(score_rt_agreement)
df['score_spectrum'] = df['entropy'].apply(score_spectrum_quality)

print(f'\nComponent summary:')
for col in ['score_ms2', 'score_unique', 'score_rt', 'score_spectrum']:
    print(f'  {col}: mean={df[col].mean():.3f}  median={df[col].median():.3f}  min={df[col].min():.3f}')

## 3. Combine into confidence score

In [ ]:
# Geometric mean (multiplicative): if any component is low, score collapses
# Use geometric mean: (a * b * c * d)^(1/4)
df['confidence_score'] = (
    df['score_ms2'] * 
    df['score_unique'] * 
    df['score_rt'] * 
    df['score_spectrum']
) ** 0.25

# Flag "absurd" identifications: score < 0.3 OR any single component < 0.1
df['is_absurd'] = (
    (df['confidence_score'] < 0.3) |
    (df['score_ms2'] < 0.1) |
    (df['score_unique'] < 0.1) |
    (df['score_rt'] < 0.1) |
    (df['score_spectrum'] < 0.1)
)

print(f'\nConfidence score distribution:')
print(f'  mean: {df["confidence_score"].mean():.3f}')
print(f'  median: {df["confidence_score"].median():.3f}')
print(f'  std: {df["confidence_score"].std():.3f}')
print(f'  min: {df["confidence_score"].min():.3f}')
print(f'  max: {df["confidence_score"].max():.3f}')
print(f'\nAbsurd identifications: {df["is_absurd"].sum()} ({df["is_absurd"].mean()*100:.1f}%)')

# Show examples of absurd cases
print(f'\nTop 10 most absurd identifications:')
absurd_sorted = df[df['is_absurd']].sort_values('confidence_score')
for _, row in absurd_sorted.head(10).iterrows():
    reason = []
    if row['score_ms2'] < 0.1:
        reason.append('low MS2')
    if row['score_unique'] < 0.1:
        reason.append('no uniqueness')
    if row['score_rt'] < 0.1:
        reason.append('RT way off')
    if row['score_spectrum'] < 0.1:
        reason.append('low entropy')
    print(f'  {row["name"]:<40s} score={row["confidence_score"]:.3f}  [{" | ".join(reason)}]')

## 4. Validate against Oliver's probability and comments

In [ ]:
# Correlation with Oliver's probability
valid = df[df['oliver_prob'].notna()].copy()
corr = valid['confidence_score'].corr(valid['oliver_prob'])
print(f'Correlation with Oliver\'s probability: {corr:.3f}')

# Stratify by confidence and show Oliver's mean probability
print(f'\nMean Oliver probability by confidence tier:')
for tier, (low, high, label) in enumerate([
    (0.0, 0.3, 'Absurd'),
    (0.3, 0.5, 'Suspicious'),
    (0.5, 0.7, 'Uncertain'),
    (0.7, 0.9, 'Confident'),
    (0.9, 1.0, 'High confidence'),
]):
    sub = valid[(valid['confidence_score'] >= low) & (valid['confidence_score'] < high)]
    if len(sub) > 0:
        print(f'  {label:<20s} ({low:.1f}–{high:.1f}): oliver_prob={sub["oliver_prob"].mean():>+.3f}  n={len(sub)}')

# How many of Oliver's ISF comments fall into each tier?
print(f'\nOliver\'s ISF cases by confidence tier:')
isf_keywords = ['isf', 'in-source', 'in source', 'insource', 'fragment']
df['has_isf_comment'] = df['oliver_comment'].fillna('').str.lower().apply(
    lambda c: any(kw in c for kw in isf_keywords)
)
isf_cases = df[df['has_isf_comment']]
if len(isf_cases) > 0:
    for tier, (low, high, label) in enumerate([
        (0.0, 0.3, 'Absurd'),
        (0.3, 0.5, 'Suspicious'),
        (0.5, 0.7, 'Uncertain'),
        (0.7, 1.0, 'Confident+'),
    ]):
        sub = isf_cases[(isf_cases['confidence_score'] >= low) & (isf_cases['confidence_score'] < high)]
        print(f'  {label:<20s}: {len(sub)}/{len(isf_cases)}')

## 5. Visualize confidence distribution and correlation with Oliver

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Confidence histogram
ax = axes[0, 0]
ax.hist(df['confidence_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.7)
ax.axvline(0.3, color='red', ls='--', lw=2, label='Absurd threshold')
ax.axvline(0.5, color='orange', ls='--', lw=2, label='Suspicious threshold')
ax.set_xlabel('Confidence score')
ax.set_ylabel('Count')
ax.set_title('Confidence score distribution')
ax.legend()

# Panel 2: Confidence vs Oliver probability
ax = axes[0, 1]
valid = df[df['oliver_prob'].notna()]
ax.scatter(valid['confidence_score'], valid['oliver_prob'], alpha=0.2, s=5, c='steelblue')
# Add binned means
bins = np.linspace(0, 1, 11)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_means = []
for i in range(len(bins)-1):
    sub = valid[(valid['confidence_score'] >= bins[i]) & (valid['confidence_score'] < bins[i+1])]
    if len(sub) > 0:
        bin_means.append(sub['oliver_prob'].mean())
    else:
        bin_means.append(np.nan)
ax.plot(bin_centers, bin_means, 'ro-', lw=2, markersize=8, label=f'Correlation: {corr:.3f}')
ax.set_xlabel('Confidence score')
ax.set_ylabel('Oliver probability')
ax.set_title('Confidence vs Oliver\'s expert judgment')
ax.axhline(0, color='gray', ls='-', lw=0.5, alpha=0.3)
ax.legend()

# Panel 3: Component breakdown (violin plots)
ax = axes[1, 0]
components = ['score_ms2', 'score_unique', 'score_rt', 'score_spectrum']
data_to_plot = [df[col].dropna() for col in components]
parts = ax.violinplot(data_to_plot, positions=range(len(components)), showmeans=True)
ax.set_xticks(range(len(components)))
ax.set_xticklabels(['MS2', 'Unique', 'RT', 'Spectrum'], rotation=45)
ax.set_ylabel('Component score')
ax.set_title('Component score distributions')
ax.set_ylim(0, 1)
ax.axhline(0.1, color='red', ls='--', lw=0.5, alpha=0.3, label='Absurd threshold')
ax.legend()

# Panel 4: Absurd vs non-absurd
ax = axes[1, 1]
for is_abs, color, label in [(True, 'crimson', 'Absurd'), (False, 'forestgreen', 'Not absurd')]:
    sub = df[df['is_absurd'] == is_abs]
    ax.hist(sub['confidence_score'], bins=30, alpha=0.5, color=color, label=label, edgecolor='white')
ax.set_xlabel('Confidence score')
ax.set_ylabel('Count')
ax.set_title('Absurd vs non-absurd identifications')
ax.legend()

plt.tight_layout()
plt.savefig(f'{ROOT}/figures/confidence_score_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure → figures/confidence_score_analysis.png')

## 6. Export confidence scores and absurd flagged cases

In [ ]:
# Full export
export_cols = [
    'wiki_id', 'name', 'adduct', 'precursor_mz', 'rt',
    'identity', 'fuzzy', 'entropy', 'sim_gap', 'delta_rt_pred',
    'score_ms2', 'score_unique', 'score_rt', 'score_spectrum',
    'confidence_score', 'is_absurd',
    'oliver_prob', 'oliver_comment',
]

out_df = df[[c for c in export_cols if c in df.columns]].copy()
out_df = out_df.sort_values('confidence_score')
out_df.to_csv(f'{OUT}/confidence_scores.csv', index=False)
print(f'Saved full scores → {OUT}/confidence_scores.csv')

# Absurd cases only
absurd_export = out_df[out_df['is_absurd']].sort_values('confidence_score')
absurd_export.to_csv(f'{OUT}/absurd_identifications.csv', index=False)
print(f'Saved {len(absurd_export)} absurd cases → {OUT}/absurd_identifications.csv')

# Summary
print(f'\n=== Summary ===')
print(f'Total spectra: {len(df)}')
print(f'Absurd identifications: {df["is_absurd"].sum()} ({df["is_absurd"].mean()*100:.1f}%)')
print(f'Suspicious (score 0.3-0.5): {((df["confidence_score"] >= 0.3) & (df["confidence_score"] < 0.5)).sum()}')
print(f'Uncertain (score 0.5-0.7): {((df["confidence_score"] >= 0.5) & (df["confidence_score"] < 0.7)).sum()}')
print(f'Confident (score > 0.7): {(df["confidence_score"] >= 0.7).sum()}')